# 08 · QA Summary

**What this does:** One page that shows what is in the pipeline right now: how many titles, how much media spend and premium revenue, and how much of the spend already has a channel. It is a check before the data goes to the model.

| | |
|---|---|
| **When to run** | Any time; best after 06_Build_Core and after a channel review is saved in 07_Review_Channels |
| **How long** | Under a minute |
| **What it changes** | Nothing in BigQuery. It only reads. Step 8 saves the title table as a CSV file in the Domino dataset (folder `us_hment_mmm/qa`). |
| **What is counted** | Titles released in 2023 or later (the MMM's titles). Home Entertainment media only (clients ZUE and ZUV; the Focus theatrical campaigns under ZFF and ZFE are shown once, for reference). Media spend that has run (not spend that is only booked) and is linked to a film. Channels are the labels saved in 07 at the moment you run this, so a new channel review shows up here on the next run. |

### How to use it
Click **Run ▸ Run All Cells** and read from top to bottom. If a number looks wrong, save the notebook and send it to the pipeline owner.

## Step 1 · Load the data

In [ ]:
from IPython.display import Markdown, display

from us_hment_mmm.qa import summary as qa

try:
    data = qa.load()
except Exception as exc:
    display(Markdown(f"### 🛑 STOP — the data could not be read\n\n`{exc}`\n\nNothing was "
                     "changed. Save this notebook and send it to the pipeline owner."))
    raise

display(Markdown("✅ Data loaded."))

## Step 2 · The numbers

In [ ]:
display(qa.headline_markdown(qa.headline(data)))

## Step 3 · Media spend by channel
Grey is spend that still needs a channel in 07_Review_Channels. Removed placements are left out.

In [ ]:
display(qa.chart_channels(data))

## Step 4 · Media spend by month
All media (Prisma, NetPak and SpotPak) against premium revenue (PVOD to EST − 1), by month, from 2023. **Hover over a month** to see both amounts; click a line in the legend to hide or show it. Premium revenue is counted in the month it was earned; pre-sales count in the PVOD month.

In [ ]:
display(qa.chart_monthly(data))

## Step 5 · Media spend vs premium revenue
One dot per title, coloured by distributor. **Hover over a dot** to see the title's details; click a distributor in the legend to hide or show it. Both scales are logarithmic, so each grid line is ten times the one before. Titles far from the cloud are worth a look (missing media, or revenue linked to the wrong title).

In [ ]:
display(qa.chart_spend_vs_revenue(data))

## Step 6 · Channel labels on the biggest titles
Blue is spend with a channel, grey still needs one.

In [ ]:
display(qa.chart_channel_coverage(data))

## Step 7 · Premium revenue by retailer

In [ ]:
display(qa.chart_retailers(data))

## Step 8 · Every title
Sorted by release date, oldest first. Use the scroll bar to see all columns.

In [ ]:
display(qa.film_table(data))

**Save this table as a CSV** (with each title's IMDb ID). Run the cell below; it says where the file is.

In [ ]:
display(qa.save_film_csv(data))

### What the columns mean

| Term | Meaning |
|---|---|
| **Premium revenue** | US royalty from the **PVOD date through the day before EST**. Sales before the PVOD date (pre-sales) are counted on the PVOD day. Amazon comes from Amazon Royalty only. This is the revenue the MMM models. |
| **Blank premium revenue** | The title has no PVOD date (it went straight to EST), or no sales are linked to it yet. |
| **Revenue after EST** | US royalty from the EST date onward. Shown for comparison; not used by the MMM. |
| **Media spend** | Home Entertainment spend (clients ZUE and ZUV) that has already run (not spend that is only booked), for media linked to the title. Focus theatrical campaigns (ZFF, ZFE) are not included. Prisma is digital and print; TV / audio / MVPD is NetPak and SpotPak. |
| **No channel yet** | Media spend that still needs a channel label in 07_Review_Channels. |
| **Premium revenue per $1 media** | Premium revenue divided by media spend. A quick check, not a measure of what the media earned. |
| **Client** | The Home Entertainment client codes that bought media for the title: ZUE (Prisma, TV) and ZUV (MVPD). |
| **In MMM** | Whether the title is used in the model (decided in 05_Review_Titles). |

---
This notebook only reads (and saves the CSV), so it is safe to run as a Domino Job.